In [1]:
import html
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

warnings.filterwarnings("ignore", message=".*BeautifulSoup.*")
pd.set_option("display.max_colwidth", 120)


C:\Users\Vivobook\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv("train_data.csv", low_memory=False)
test_data = pd.read_csv("test_data.csv", low_memory=False)
title_brand = pd.read_csv("title_brand.csv", low_memory=False)

print(f"Train: {train_data.shape}; test: {test_data.shape}; catalog: {title_brand.shape}")


Train: (838944, 11); test: (20000, 10); catalog: (786445, 3)


In [3]:
if "asin" not in title_brand.columns:
    raise KeyError("title_brand.csv must contain an 'asin' column.")

duplicate_count = title_brand["asin"].duplicated().sum()
title_brand_unique = title_brand.drop_duplicates(subset="asin", keep="first").copy()
print(f"Removed {duplicate_count:,} duplicate catalog rows; kept {len(title_brand_unique):,} rows.")


Removed 30,368 duplicate catalog rows; kept 756,077 rows.


In [4]:
def merge_catalog(frame):
    if "asin" not in frame.columns:
        raise KeyError("Input data must contain an 'asin' column.")

    original_rows = len(frame)
    merged = frame.merge(
        title_brand_unique,
        on="asin",
        how="left",
        validate="many_to_one",
        suffixes=("", "_catalog"),
    )
    if len(merged) != original_rows:
        raise AssertionError("Catalog merge unexpectedly changed the row count.")

    for column in title_brand_unique.columns.difference(["asin"]):
        catalog_column = f"{column}_catalog"
        if catalog_column in merged.columns:
            merged[column] = merged[catalog_column].combine_first(merged[column])
            merged = merged.drop(columns=catalog_column)
    return merged

train_data = merge_catalog(train_data)
test_data = merge_catalog(test_data)
print(f"Merged train: {train_data.shape}; merged test: {test_data.shape}")


Merged train: (838944, 13); merged test: (20000, 12)


In [5]:
HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"(?:http\S+|www\.\S+)", flags=re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")
TEXT_COLUMNS = ("reviewText", "summary", "title")

def clean_text_columns(frame, columns=TEXT_COLUMNS):
    """Decode entities and remove markup, URLs, and redundant whitespace."""
    result = frame.copy()
    for column in columns:
        if column not in result.columns:
            result[column] = ""
        text = result[column].fillna("").astype("string")
        text = text.map(html.unescape)
        text = text.str.replace(HTML_TAG_RE, " ", regex=True)
        text = text.str.replace(URL_RE, " ", regex=True)
        text = text.str.replace(WHITESPACE_RE, " ", regex=True).str.strip()
        result[column] = text.fillna("")
    return result


In [6]:
train_data = clean_text_columns(train_data)
test_data = clean_text_columns(test_data)


In [7]:
# ---------------------------------------------------------------------------
# INPUT_MODE decides which fields the model sees. Run the training notebook once per mode and compare:
#   "full"                 -> Brand | Title | Summary | Review      (the old v2 behaviour)
#   "summary_review"       -> Summary | Review                      (brand/title are mostly noise for sentiment)
#   "title_summary_review" -> Title | Summary | Review
# ---------------------------------------------------------------------------
INPUT_MODE = "summary_review"
FIELDS_BY_MODE = {
    "full": ["brand", "title", "summary", "review"],
    "summary_review": ["summary", "review"],
    "title_summary_review": ["title", "summary", "review"],
}
LABELS = {"brand": "Brand", "title": "Title", "summary": "Summary", "review": "Review"}

def build_model_input(frame, mode=INPUT_MODE, summary_max_words=40):
    """Combine fields without altering case or punctuation.
    Rows that would end up empty in `mode` fall back to the full input, so the set of usable rows
    (and therefore the validation split) is identical for every mode."""
    def text_column(name):
        if name in frame.columns:
            return frame[name].fillna("").astype("string").str.strip().fillna("")
        return pd.Series("", index=frame.index, dtype="string")

    cols = {
        "brand": text_column("brand"),
        "title": text_column("title"),
        "summary": text_column("summary").str.split().str[:summary_max_words].str.join(" "),
        "review": text_column("reviewText"),
    }

    def join(fields):
        parts = [cols[n].where(cols[n].eq(""), f"{LABELS[n]}: " + cols[n]) for n in fields]
        out = parts[0]
        for p in parts[1:]:
            out = out.str.cat(p, sep=" | ")
        return out.str.strip(" |")

    full = join(FIELDS_BY_MODE["full"])
    chosen = join(FIELDS_BY_MODE[mode])
    return chosen.where(chosen.ne(""), full)

train_data["model_input"] = build_model_input(train_data)
test_data["model_input"] = build_model_input(test_data)
train_data = train_data.loc[train_data["model_input"].ne("")].copy()
print(f"INPUT_MODE = {INPUT_MODE!r}; train rows with usable text: {len(train_data):,}")
print(train_data["model_input"].iloc[0][:300])

INPUT_MODE = 'summary_review'; train rows with usable text: 838,944
Summary: Cannot Learn | Review: I have an older URC-WR7 remote and thought this would be an upgrade (and because TWC stuck me with it), but this one fails where the other one didn't. The old remote could go head to head and learn. I have 2 different Blu Ray players (LG & Panasonic) and this one fail


In [8]:
if "overall" not in train_data.columns:
    raise KeyError("Training data must contain the 'overall' target column.")

labeled_data = train_data.loc[train_data["overall"].notna()].copy()
class_counts = labeled_data["overall"].value_counts().sort_index()
print("Full labeled class counts:")
print(class_counts)
if class_counts.empty or class_counts.min() < 2:
    raise ValueError("Each target class needs at least two rows for stratified splitting.")

# --- Same sampling + split as before, so val_processed.csv is IDENTICAL to the old one
#     (the index is kept so we can find these rows again; train_test_split only uses positions).
samples_per_class = min(30_000, int(class_counts.min()))
balanced_data = (
    labeled_data.groupby("overall", group_keys=False, sort=True)
    .sample(n=samples_per_class, random_state=42)
)
_, val_df = train_test_split(
    balanced_data,
    test_size=0.20,
    random_state=42,
    stratify=balanced_data["overall"],
)

# --- NEW: the training POOL = every labeled row that is not in validation.
#     The training notebook draws a fresh balanced subset from this pool each epoch.
pool_df = labeled_data.drop(index=val_df.index)

# Leakage guard: exact-duplicate reviews are common on Amazon (same text under several ASINs).
val_texts = set(val_df["model_input"])
before = len(pool_df)
pool_df = pool_df.loc[~pool_df["model_input"].isin(val_texts)]
removed_overlap = before - len(pool_df)
before = len(pool_df)
pool_df = pool_df.drop_duplicates(subset="model_input")
removed_dups = before - len(pool_df)
print(f"Removed from pool: {removed_overlap:,} rows duplicated in validation, {removed_dups:,} duplicates inside the pool")

train_texts = set(pool_df["model_input"])
test_overlap = test_data["model_input"].isin(train_texts).sum()
print(f"Test rows whose exact text also appears in the pool: {test_overlap:,}")

pool_df = pool_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_data.reset_index(drop=True)

print("Pool rows per class:")
print(pool_df["overall"].value_counts().sort_index())
print("Validation rows per class:")
print(val_df["overall"].value_counts().sort_index())
print(f"Pool: {len(pool_df):,}; validation: {len(val_df):,}; test: {len(test_df):,}")


Full labeled class counts:
overall
1     82950
2     56756
3     81239
4    156514
5    461485
Name: count, dtype: int64
Removed from pool: 2,528 rows duplicated in validation, 27,872 duplicates inside the pool
Test rows whose exact text also appears in the pool: 1,120
Pool rows per class:
overall
1     73754
2     48827
3     72368
4    145285
5    438310
Name: count, dtype: int64
Validation rows per class:
overall
1    6000
2    6000
3    6000
4    6000
5    6000
Name: count, dtype: int64
Pool: 778,544; validation: 30,000; test: 20,000


In [9]:
TAG = "" if INPUT_MODE == "full" else f"_{INPUT_MODE}"   # the training notebook's DATA_TAG must equal this
cols_to_save_train = ["model_input", "overall"]
cols_to_save_test = ["model_input"]

pool_df[cols_to_save_train].to_csv(f"pool_processed{TAG}.csv", index=False, encoding="utf-8")
val_df[cols_to_save_train].to_csv(f"val_processed{TAG}.csv", index=False, encoding="utf-8")
test_df[cols_to_save_test].to_csv(f"test_processed{TAG}.csv", index=False, encoding="utf-8")

print(f"Saved pool/val/test_processed{TAG}.csv  ->  set DATA_TAG = {TAG!r} in the training notebook.")

Saved pool/val/test_processed_summary_review.csv  ->  set DATA_TAG = '_summary_review' in the training notebook.


In [10]:
train_data.head()

,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,title,brand,model_input
0,2,NaN,False,2016-11-11,A2OSUEZJIN7BI,0511189877,NaN,Chris,"I have an older URC-WR7 remote and thought this would be an upgrade (and because TWC stuck me with it), but this one...",Cannot Learn,1478822400,CLIKR-5 Time Warner Cable Remote Control UR5U-8780L,URC,Summary: Cannot Learn | Review: I have an older URC-WR7 remote and thought this would be an upgrade (and because TWC...
1,5,NaN,True,2016-06-06,A2NETQRG6JHIG7,0511189877,NaN,Qrysta White,First time I've EVER had a remote that needed no programming whatsoever so I have to give it bragging rights. All th...,zero programming needed! Miracle!?,1465171200,CLIKR-5 Time Warner Cable Remote Control UR5U-8780L,URC,Summary: zero programming needed! Miracle!? | Review: First time I've EVER had a remote that needed no programming w...
2,4,NaN,True,2016-03-10,A12JHGROAX49G7,0511189877,NaN,Linwood,Got them and only 2 of them worked. company called me after the feedback and told me they would send me a refund. Th...,Works Good and programs easy.,1457568000,CLIKR-5 Time Warner Cable Remote Control UR5U-8780L,URC,Summary: Works Good and programs easy. | Review: Got them and only 2 of them worked. company called me after the fee...
3,5,NaN,True,2016-01-14,A1KV65E2TMMG6F,0511189877,NaN,Dane Williams,I got tired of the remote being on the wrong side of the bed. I was going to buy a generic remote when I got the ide...,Same as TWC remote,1452729600,CLIKR-5 Time Warner Cable Remote Control UR5U-8780L,URC,Summary: Same as TWC remote | Review: I got tired of the remote being on the wrong side of the bed. I was going to b...
4,5,NaN,True,2016-10-20,A280POPEWI0NSA,0594459451,NaN,Kristina H.,"After purchasing cheap cords from another website, I was a little scared to purchase. So glad I did though! It works...",Good Quality Cord,1476921600,Barnes & Noble Nook Color Tablet USB Cable Charger Newest Re-enforced Version,Barnes &amp; Noble,"Summary: Good Quality Cord | Review: After purchasing cheap cords from another website, I was a little scared to pur..."
